In [1]:
import pandas as pd
import os

RAW_DATA_PATH = "../data/raw"
REPORT_PATH = "../reports"

os.makedirs(REPORT_PATH, exist_ok=True)

datasets = {}

for file in os.listdir(RAW_DATA_PATH):
    if file.endswith(".csv"):
        dataset_name = file.replace(".csv", "")
        file_path = os.path.join(RAW_DATA_PATH, file)

        datasets[dataset_name] = pd.read_csv(file_path)

print("Total datasets loaded:", len(datasets))

for name, df in datasets.items():
    print(name, ":", df.shape)

Total datasets loaded: 13
process_parameters : (121, 7)
material_lots : (12, 8)
machines : (6, 4)
products : (4, 5)
rework_records : (15, 9)
suppliers : (4, 4)
operators : (8, 4)
production_lines : (3, 4)
shift_schedule : (3, 3)
batch_relationships : (14, 5)
production_batches : (41, 8)
production_events : (201, 8)
defects : (37, 9)


In [4]:
datasets["defects"]

,defect_id,batch_id,inspection_id,defect_category,defect_code,affected_quantity,inspected_quantity,detected_at,severity
0,DF-00001,BAT-0001,INSP-0001,CRACK,D03,-3,150,2026-09-01 21:00:00,LOW
1,DF-00002,BAT-5555,INSP-0002,SURFACE,D01,1,200,2026-09-01 20:00:00,HIGH
2,DF-00003,BAT-0002,INSP-0002,unknown defect,D04,5,100,2026-09-01 21:00:00,LOW
3,DF-00004,BAT-0003,INSP-0003,DIMENSIONAL,D05,3,0,2026-09-02 00:00:00,LOW
4,DF-00005,BAT-0004,INSP-0004,POROSITY,D02,3,150,32-09-2026,MEDIUM
5,DF-00006,BAT-0005,INSP-0005,DIMENSIONAL,D04,2,200,2026-09-02 18:00:00,URGENT
6,DF-00007,BAT-0005,INSP-0005,POROSITY,NaN,10,200,2026-09-02 19:00:00,MEDIUM
7,DF-00008,BAT-0006,INSP-0006,CONTAMINATION,D01,2,80,2026-09-02 21:00:00,HIGH
8,DF-00009,BAT-0008,INSP-0008,POROSITY,D04,5,150,2026-09-03 18:00:00,LOW
9,DF-00010,BAT-0009,INSP-0009,POROSITY,D01,8,80,2026-09-03 16:00:00,LOW


In [5]:
for name, df in datasets.items():
    print(f"\n{'=' * 60}")
    print("DATASET:", name)
    print("ROWS:", df.shape[0])
    print("COLUMNS:", df.shape[1])
    print("\nCOLUMN NAMES:")
    print(df.columns.tolist())
    print("\nDATA TYPES:")
    print(df.dtypes)


DATASET: process_parameters
ROWS: 121
COLUMNS: 7

COLUMN NAMES:
['measurement_id', 'batch_id', 'machine_id', 'parameter_name', 'parameter_value', 'unit', 'recorded_at']

DATA TYPES:
measurement_id         str
batch_id               str
machine_id             str
parameter_name         str
parameter_value    float64
unit                   str
recorded_at            str
dtype: object

DATASET: material_lots
ROWS: 12
COLUMNS: 8

COLUMN NAMES:
['material_lot_id', 'material_type', 'supplier_id', 'supplier_name', 'received_date', 'certificate_status', 'quantity_received', 'unit']

DATA TYPES:
material_lot_id         str
material_type           str
supplier_id             str
supplier_name           str
received_date           str
certificate_status      str
quantity_received     int64
unit                    str
dtype: object

DATASET: machines
ROWS: 6
COLUMNS: 4

COLUMN NAMES:
['machine_id', 'machine_name', 'machine_type', 'machine_status']

DATA TYPES:
machine_id        str
machine_name  

In [6]:
for name, df in datasets.items():
    missing = df.isnull().sum()

    missing = missing[missing > 0]

    print(f"\nDATASET: {name}")

    if missing.empty:
        print("No missing values found.")
    else:
        print(missing)


DATASET: process_parameters
parameter_value    1
dtype: int64

DATASET: material_lots
material_type    1
supplier_id      1
dtype: int64

DATASET: machines
No missing values found.

DATASET: products
No missing values found.

DATASET: rework_records
No missing values found.

DATASET: suppliers
No missing values found.

DATASET: operators
No missing values found.

DATASET: production_lines
No missing values found.

DATASET: shift_schedule
No missing values found.

DATASET: batch_relationships
parent_batch_id    1
dtype: int64

DATASET: production_batches
batch_id      1
product_id    1
dtype: int64

DATASET: production_events
material_lot_id    160
quantity           160
operator_id          1
dtype: int64

DATASET: defects
defect_code    1
dtype: int64


In [7]:
for name, df in datasets.items():
    exact_duplicates = df.duplicated().sum()

    print(
        f"{name}: "
        f"{exact_duplicates} exact duplicate rows"
    )

process_parameters: 1 exact duplicate rows
material_lots: 0 exact duplicate rows
machines: 0 exact duplicate rows
products: 0 exact duplicate rows
rework_records: 1 exact duplicate rows
suppliers: 0 exact duplicate rows
operators: 0 exact duplicate rows
production_lines: 0 exact duplicate rows
shift_schedule: 0 exact duplicate rows
batch_relationships: 0 exact duplicate rows
production_batches: 1 exact duplicate rows
production_events: 1 exact duplicate rows
defects: 1 exact duplicate rows


In [8]:
categorical_columns = [
    "status",
    "shift",
    "event_type",
    "defect_category",
    "severity",
    "rework_outcome",
    "rework_type",
    "relationship_type",
    "certificate_status",
    "supplier_status",
    "machine_status",
    "line_status"
]

for name, df in datasets.items():
    for col in categorical_columns:
        if col in df.columns:
            print(f"\n{name} -> {col}")
            print(df[col].value_counts(dropna=False))


material_lots -> certificate_status
certificate_status
APPROVED    6
PENDING     5
UNKNOWN     1
Name: count, dtype: int64

machines -> machine_status
machine_status
ACTIVE    6
Name: count, dtype: int64

rework_records -> rework_outcome
rework_outcome
PASSED     5
FAILED     5
PENDING    4
DONE??     1
Name: count, dtype: int64

rework_records -> rework_type
rework_type
REPAIR       6
REPROCESS    5
REPLACE      4
Name: count, dtype: int64

suppliers -> supplier_status
supplier_status
ACTIVE    4
Name: count, dtype: int64

production_lines -> line_status
line_status
ACTIVE    3
Name: count, dtype: int64

shift_schedule -> shift
shift
DAY        1
EVENING    1
NIGHT      1
Name: count, dtype: int64

batch_relationships -> relationship_type
relationship_type
SPLIT               9
MERGE               4
UNKNOWN_RELATION    1
Name: count, dtype: int64

production_batches -> status
status
COMPLETED     21
REWORK        10
HOLD           9
FINISHED??     1
Name: count, dtype: int64

product